<a href="https://colab.research.google.com/github/IsabellaHannaCSM/skills-introduction-to-github/blob/main/05_Clustering_evaluation_vic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **BiotrAIn 2025 Tutorial: Data integration for microbiome research using scikit-bio**


# Section 05: Clustering Evaluation (40 min)

- Time: 13:00 - 15:30 EDT, June 17, 2025
Welcome to the practical session 01. created by Víctor Muñiz, Leticia Ramírez, Nelly Selem (Secodment host), Jeanett Daga (Secodment), with modifications by professor Qiyun Zhu (on behalf of Daniel McDonald).

🏆 QUESTIONS

How to choose the clustering evaluation method?


🎯 AIMS

To analyze and compare the microbial community composition between samples from different environments or conditions using beta diversity metrics.

To apply UniFrac (both weighted and unweighted) as a phylogeny-based distance measure for assessing differences in microbial communities.

To visualize these differences through Non-Metric Multidimensional Scaling (NMDS) ordination plots.

🔑 KEY POINTS

Beta diversity

UniFrac

NMDS

Non-metric multidimensional scaling (NMDS)


### Background

Beta diversity measures the differences in microbial community composition between different environments or samples. It captures how distinct or similar microbial communities are across spatial or environmental gradients.

In this workshop section, we will focus on beta-diversity using:

1. Bray-Curtis dissimilarity (based on abundance data) and Jaccard index (based on presence/absence), Heatmaps to show pairwise dissimilarity or abundance patterns.

2. UniFrac, on shotgun data from the EMP500 dataset. We will apply UniFrac which is a popular phylogenetic beta-diversity metric. The similarity computed is the **uni**que **frac**tion of the phylogenetic branch length between a pair of samples.

3. Principal Coordinates Analysis (PCoA): Visualizes multivariate dissimilarities in 2D or 3D space. Samples closer together have more similar communities, and Non-metric Multidimensional Scaling (NMDS): Another ordination method for representing similarity patterns.

4. Apply PERMANOVA (Permutational Multivariate Analysis of Variance) to test for significant differences among groups of samples.

5. Last, to further explore the capabilities of scikit-bio, we will  test whether sample similarities are correlated between two prevalent phyla using both a Mantel test and Procrustes analysis.

## Preparation (5 min)

Install the latest version of scikit-bio if it hasn't been (needed for every Google Colab instance).

In [ ]:
from importlib.util import find_spec

In [ ]:
if find_spec('skbio') is None:
    !pip install -q scikit-bio

In [ ]:
import skbio
skbio.__version__

### Importing auxiliary functions

<!--

We will be using different functions available in the script file ``aux_functions.py``. In order to use them, we need to load them from local drive by running the following code:[texto del vínculo](https:// [texto del vínculo](https://))

-->

We will be using different functions available in the script file ``aux_functions.py``. In order to use them, we need to load them from GitHub by running the following code:

In [ ]:
data_commit = "48d9e293bcccfa31f2d6e272641136753d4e5e5f"
code_commit = "b74c2d94ab8fdcaa6807aefa5c31c13793412115"
!wget "https://github.com/nselem/biotraintemp/archive/{data_commit}.zip"
!unzip -q "{data_commit}.zip"
!mkdir "biotraintemp-{data_commit}/code"
%cd "biotraintemp-{data_commit}/code"
!wget "https://raw.githubusercontent.com/biotrain-latam/BiotrAIn-pilot-course/{code_commit}/Module%202%20Clustering/code/aux_functions.py"
!ln -s latam ../data/Data_Latam
!mkdir "../models"

<!--
```python
from google.colab import drive
import os
drive.mount('/content/drive/')
# working directory
os.chdir('/content/drive/My Drive/Module 2 Clustering/2. Notebooks_Latam/')
```
-->

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from skbio import Table
from skbio.diversity import beta_diversity
import matplotlib.pyplot as plt
import plotly.express as px

## Loading and preparing the data (5 min)



We are using the following data to make a Beta biodiversity analysis To compute and visualize beta diversity, you need:

1. Sample metadata (latam_samples.tsv): environmental metadata per sample.
2. feature table (latam_ogu.biom)counts of microbial taxa (OGUs) per sample.
3. Phylogenetic tree (latam_ogu.nwk)for UniFrac.
4. Taxonomy Taxonomy assignment file (latam_ogu.tax). for heatmaps

In [ ]:
from aux_functions import get_data

# Create a text variable to use after as a path to read files
# github_data = "https://raw.githubusercontent.com/nselem/biotraintemp/refs/heads/main/data/"
github_data = "../data/"
# we will consider just Latinamerican data
biom_df, metadata_df = get_data(github_data)

In [ ]:
# labels for each sample
labels_data = metadata_df['empo_3']
labels_data.head(5)

## Fitting one clustering model (5 min )

QUE HACE LA SIG CELDA

In [ ]:
from aux_functions import get_distance_matrix, get_tfidf, get_lsa, get_nmf, hierarchical_cluster

tfidf_vect, tfidf = get_tfidf(biom_df)
biom_tfidf = pd.DataFrame(tfidf,columns=tfidf_vect.get_feature_names_out(),index=biom_df.index)

### Manifold learning method

#### Model 1

QUE HACE ESTA CELDA REDUCCION DE DIMENSION?

In [ ]:
import umap.umap_ as umap

reducer = umap.UMAP(n_neighbors=8, n_components=3, min_dist=.12, metric='braycurtis',
                    random_state=42)
embedding = reducer.fit_transform(biom_tfidf)

CLUSTERIN?

In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans
nclust = len(labels_data.unique())  # we set the number of cluster the same as the number of different labels

X = embedding
kmeans = KMeans(n_clusters = nclust, init = 'k-means++', n_init = 'auto', random_state = 42)
y_km = kmeans.fit_predict(X)
#y_km = y_km.astype('category')

VISUALIZACION?

We graph the resulting clusters

In [ ]:
reduced_proj = pd.DataFrame({'x1': embedding[:,0], 'x2': embedding[:,1], 'label': labels_data,
                            'cluster': [str(val) for val in y_km]})
#reduced_proj['cluster'].astype('category')
fig = px.scatter(reduced_proj, x='x1', y='x2', hover_data='label', color = 'cluster', title='UMAP')
fig.update_layout(autosize=False, width=500, height=500)
fig.show()

## Selecting the number of groups (15 min)

Up to this point, we have assumed that the number of clusters matches the number of true classes in the data. However, it may be beneficial to merge certain groups or introduce additional ones to achieve a more meaningful separation. Selecting the optimal number of clusters is a critical step in any unsupervised learning process and can also enhance supervised learning by revealing significant group structures within the data. In this section, we review two important methods for determining the ideal number of clusters: the Elbow method and the Silhouette score.

### Elbow

The Elbow Method is a popular technique used for this purpose in K-Means clustering and tipically consist in the following steps:

- Ror a specific value of k (number of clusters) we calculate a distance measure called  **Within-Cluster Sum of Squares** (WCSS). This assess how  spread out the data points are within each cluster.
- We repeat the previous step for different k values.
- We plot a graph with k on the X-axis and WCSS(k) on the Y-axis.
- Based on the plot, we identify the Elbow Point.



For a given $k$, the WCSS is:
$$
\text{WCSS}=\sum_{i=1}^k \sum_{y \in C_i}\left[y-\mu_i\right]^2
$$
where $C_i$ is the $i$-th cluster set $(i=1,\ldots,k)$, $y$ is one of its observation ($y \in C_i$) and $\mu_i$ is it centroid.

As we increase the value of k, the WCSS typically decreases because we're creating more clusters, which tend to capture more  and more data variations. However, typically,  there is a point where adding more clusters results, just marginally decrease in WCSS. This is where we observe an "elbow" shape.

<img src="https://raw.githubusercontent.com/leticiaram/figures/main/elbow_w.png" alt="Representation of elbow" width="400"/>

### Silhoutte score

Similarly to the WCSS, the Silhouette score is a metric used to evaluate how good clustering results are in data clustering.

This score is calculated by measuring each data point’s similarity to the cluster it belongs to and how different it is from other clusters (within and between distance).

1. For each data point, the average distance (a_i) to other data points within the same cluster is calculated. This value represents the **similarity** level of the data point to others in its cluster.

2. For each data point, the average distance (b_i) to all other clusters it doesn’t belong to is computed. This value indicates how **different** the data point is from data points in other clusters.

3. The cluster Silhouette score is calculated using the formula:
$$
\text{SS}(i) = \frac{b_i - a_i}{\max(a_i, b_i)}
$$
4. By taking the average of the Silhouette scores calculated for each data point, an overall Silhouette score is obtained, which measures the success of clustering results.
$$
\text{Silhouette Score}=\frac{\sum_{i=i}^k SS(i)}{k}.
$$

The Silhouette score is commonly used to assess the performance of clustering algorithms and like the elbos method can help to identify an optimum number of clusters.


In contrast to the
not supervised
is not label dependent

Can help to identify groups that be completely different from the original groups




## Metrics to evaluate the assigned group (prediction) (10 min)

In this section, we explore several metrics used to evaluate the quality of group assignments (predictions) produced by clustering methods. We assume that the true group labels and the predicted group labels are available for each observation. The core idea is to use distance-based functions to measure how closely the predicted groupings align with the true groupings, helping us assess the accuracy of the clustering results.



The considered metrics are

1. Confusion Matrix
2. Accuracy, Sensitivity, Precision, Specificity, NPV and F1 Score
4. Root Mean Squared Error (RMSE)



### Confusion Matrix

A confusion matrix is a table that describes the performance of a classification model, or a classifier, on a set of observations for which the true values are known. Each row of the matrix represents the instances in the actual class, while each column represents the instances in the predicted class (or vice versa).

When the number classes are the same as the obtained classes, a good classifications the highest counts in the diagonal. That is, a confusion matrix that concentrates its largest values along each row near the diagonal makes it easier to quickly see how much the matrix deviates from a diagonal matrix–that is how far off it is from a perfect prediction.

In [ ]:
reduced_proj.head(5)

The confusion matrix cm has columns corresponding to the cluster codes produced by the method. However, we want these codes to match the actual groups they tend to describe. This is equivalent to finding a permutation of the confusion matrix’s columns that results in a “nice” matrix, where the highest values are concentrated along the diagonal. This alignment problem can be solved using the Hungarian algorithm, which we apply next.

In [ ]:
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

# transform to categories the values of label and cluster
labels = reduced_proj['label'].astype('category').cat.codes
clusters = reduced_proj['cluster'].astype('category').cat.codes

cm = confusion_matrix(labels, clusters)  # we generate the confusion matrix

# we apply the Hungarian algorithm to determine the best correspondance:
row_ind, col_ind = linear_sum_assignment(cm, maximize=True)

cm_sorted = cm[:, col_ind] # we sort the confusion matrix

We want to display the confusion matrix with the original labels for the true categories and assigned clusters

In [ ]:
# we now create a df for the sorted cm
labels_unique = reduced_proj['label'].astype('category').cat.categories
cm_df = pd.DataFrame(cm_sorted, index = labels_unique, columns = col_ind)

# we graph the confusion matrix as a heatmap
plt.figure(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Assigned cluster')
plt.ylabel('True group')
plt.title('Confusion Matrix (sorted)')
plt.show()

Now, we rename each assigned cluster to match the name of the group it most closely corresponds to.

In [ ]:
cm_df.columns = cm_df.index.tolist()
cm_df.columns

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Assigned cluster')
plt.ylabel('True group')
plt.title('Confusion Matrix (sorted)')
plt.show()

### F1 scores

When we have two classes (usually name as negative and positive groups), based on the confusion matrix we can define "efficient" classification metrics:

#### Two-classes case

When we have two classes (usually name as negative and positive groups), based on the confusion matrix we can define "efficient" classification metrics:


<img src="https://raw.githubusercontent.com/leticiaram/figures/main/2confusion_matrix.png" alt="2-class Confusion Matrix" width="400"/>








From the elements in the confussion matrix we can also define
- **Positive Population (PP)**, is the positive population (TP+FN)
- **Negative Population (NP**), is the positive population (FP+TN)


and some popular metrics that can help us to judge the model's performance are

- **Accuracy**: Mathematically defined as (TP+TN)/Total=(TP+TN)/(PP+NP). It tells you how often the classifier is correct in making the predictions. Generally, it is not advised to judge your model on accuracy in case of imbalanced class datasets, as you can get high accuracy just by predicting all the observations as the dominant class.
- **Precision**: It answers the question: When the classifier predicts Positive, how often is it correct? Mathematically calculated as TP/"predicted Positive"=TP/(TP+FP). A value close to 1 is desirable.
- **Sensitivity (or Recall)** measures the ability of the test to detect disease in a population of diseased individuals. It answers the question: When it is "actually Positive", how often does the classifier predict Positive?. It is defined as TP/(TP + FN) = TP/PP.
- **Specificity** assesses the ability of the test to correctly rule out the disease in a disease-free population. We define it as TN/(TN + FP) = TN / NP.
- **Negative predictive value (NPV)**. It answers the question: When the classifier predicts Negative, what is the proportion of time that it does it correct? In analogy to the Precision, it is defined as TN / (TN + FN).

As some metrics are closely related—and in some cases, one can be derived from knowing two others—it is advisable to understand the meaning of the metrics rather than using all of them indiscriminately.



<img src="https://raw.githubusercontent.com/leticiaram/figures/main/2confusion_matrix_b.png" alt="2-class Confusion Matrix-metrics" width="500"/>





In particular, one of the metrics that is gaining more popularity is the  **F1 Score**, defined as the harmonic mean of the **Recall** and **Precision**. That is, it defined as

  $$
  \text{F1}=\frac{2 \times \text{precision} \times \text{recall}}{\text{precision}+\text{recall}}.
  $$

Using some algebra we can prove that
  $$
  \text{F1}=\frac{2 \times \text{TP}}{(2\times \text{TP})+\text{FN}+\text{FP}},
  $$
so it is obvious that large values of TP and small values of both, FP and FN produces values of  F1 close to 1. On the other hand, large values of FN and FP, and small values of TP will reduce the value of F1 to become close to 0.

Be aware that the F1 score is *sensitive to small values*. This means if either precision or recall is significantly lower than the other, it will have a more pronounced impact on the F1 score.

#### Multi-classes case


Extending to classification cases beyond binary variable ('Positive', 'Negative') is achieved by treating multiclass and multilabel data as a collection of binary problems, one for each label.




**Example**

<img src="https://raw.githubusercontent.com/leticiaram/figures/main/Multiconfusion_matrix.png" alt="3-class Confusion Matrix-metrics" width="500"/>





Then, for exameple, we have

- Precision(A) = 16/19,
- Precision(B) = 1,
- Precision(C) = 11/12,
- Sensitivity(A) = 1,
- Sensitivity(B) = 17/19,
- Sensitivity(C) = 11/12,

and

- F1(A) = 2*16/((2*16)+3) = 0.9142857,
- F1(B) = 2*17/((2*17)+2) = 0.9444444,
- F1(C) = 2*11/((2*11)+3) = 0.88.






We recreate the data from the confusion matrix to obtain the F1's using python command

In [ ]:
# Data
categoria=["A","B","C"]
# Definir las categorías y sus repeticiones
repeticiones = {
    'A': 16,
    'B': 19,
    'C': 13
}
ytrue = [categoria for categoria, veces in repeticiones.items() for _ in range(veces)]
ypred = [categoria for categoria, veces in repeticiones.items() for _ in range(veces)]
ypred[16]='A'
ypred[34]='C'
ypred[35]='A'
ypred[36]='A'

cm = confusion_matrix(ytrue, ypred)
cm

In [ ]:
from sklearn.metrics import f1_score
f1_score(ytrue, ypred, average=None)  # that coincides with the previously calculated F1's

A single F1 metric can be obtained for the multi-classes case averaging (simple average) the individual F1's or by  computing the average weighted by support (the number of true instances for each label). This latter, takes into account the label imbalance in the data.


In [ ]:
F1_macro = f1_score(ytrue, ypred, average='macro')        # simple average
F1_weighted = f1_score(ytrue, ypred, average='weighted')  # weighted average

print(F1_macro)
print(F1_weighted)

Now we compute the accuracy, precision and sensitivity.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Calculate accuracy
accuracy = accuracy_score(ytrue, ypred)
print("Accuracy:", accuracy)

# Calculate precision
precision = precision_score(ytrue, ypred,  average='macro')
print("Precision:", precision)

# Calculate sensitivity (recall)
recall = recall_score(ytrue, ypred, average='macro')
print("Recall (Sensitivity):", recall)

#### Metrics for the proposed models

To create label and prediction vectors that reflect the assignment obtained via the Hungarian algorithm, we can reconstruct the data using the confusion matrix with the following function:

In [ ]:
def reconstruct_data(confusion_matrix):
    real_labels = []
    predicted_labels = []

    for real_class in range(confusion_matrix.shape[0]):
        for predicted_class in range(confusion_matrix.shape[1]):
            count = confusion_matrix[real_class, predicted_class]
            real_labels.extend([real_class] * count)
            predicted_labels.extend([predicted_class] * count)

    data = np.column_stack((real_labels, predicted_labels))
    return data


We run a small example of the function 'reconstruct_data' that produces an array with two columns, the data with actual and predicted groups, respectively:

In [ ]:
# Example usage
confusion_matrix = np.array([
    [4, 2 ],
    [1, 3 ],
])

rec_data = reconstruct_data(confusion_matrix)
print(rec_data)

In [ ]:
rdata = reconstruct_data(cm_df.to_numpy())

# accuracy
accuracy = accuracy_score(rdata[:,0], rdata[:,1])
print("Accuracy:", accuracy)
# precision
precision = precision_score(rdata[:,0], rdata[:,1],  average='macro')
print("Precision:", precision)
# sensitivity (recall)
recall = recall_score(rdata[:,0], rdata[:,1], average='macro')
print("Recall (Sensitivity):", recall)
# F1
f1 = f1_score(rdata[:,0], rdata[:,1], average=None)
print("F1 s:", f1)
plt.plot(f1)
plt.title('F1 by group')
plt.show()

### Root Mean Squared Error (RMSE)

Root Mean Square Error (RMSE, Root Mean Square Deviation) is one of the most commonly used metrics for evaluating the accuracy of predictions. It measures how far predictions deviate from the true values using the Euclidean distance.

To compute RMSE, one calculates the residual (i.e., the difference between the predicted and true value) for each data point, computes the norm of these residuals, averages them, and then takes the square root of that mean. This is:

$$
\text{RMSE} = \sqrt{\frac{\sum_{i=1}^N\left[y_{\text{actual}}(i)-y_{\text{pred}}(i)\right]^2}{N}},
$$

where  $N$ is the number of data points, $y_{\text{actual}}(i)$  is the actual group of the $i$-th element, and  $y_{\text{pred}}(i)$ is its corresponding group prediction.

RMSE is particularly common in supervised learning tasks, where ground-truth labels are available for each prediction.

In [ ]:
from sklearn.metrics import root_mean_squared_error
root_mean_squared_error(rdata[:,0], rdata[:,1])

## Self evaluation poll (10 min)

To reinforce your learning from the session “Clustering Evaluation in Microbiome Research”, please complete the short quiz below. It features multiple-choice questions on essential topics such as clustering evaluation.

🧪 Start the quiz here:
👉 [Link to Poll, Quiz 5](https://pollev.com/biotrainaicabana135)

This quick assessment will help solidify your understanding and support your next steps in microbiome data analysis, interpretation and evaluation.